# 01. 데이터 탐색 및 시각화 (EDA)

김해시 CCTV 설치위치 선정 프로젝트 - 데이터 탐색 단계

- 원본 데이터 불러오기 및 기본 정보 확인
- CCTV 설치 전후 112신고 건수 비교
- 공간 데이터 시각화

## 0. 라이브러리 및 데이터 불러오기

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import folium
from folium.plugins import HeatMap
import warnings
warnings.filterwarnings('ignore')

# 한글 폰트 설정
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# COMPAS 데이터 로드 (플랫폼 환경)
# from geoband.API import *
# GetCompasData('SBJ_2206_001', '1', 'data/raw/1.김해시_CCTV설치현황.csv')
# GetCompasData('SBJ_2206_001', '25', 'data/raw/4.김해시_112신고이력(격자매핑).csv')
# GetCompasData('SBJ_2206_001', '5', 'data/raw/5.김해시_격자(100X100).geojson')
# GetCompasData('SBJ_2206_001', '9', 'data/raw/9.김해시_하천현황.geojson')

# 로컬 환경
df_cctv   = pd.read_csv('data/raw/1.김해시_CCTV설치현황.csv', encoding='cp949')
df_police = pd.read_csv('data/raw/4.김해시_112신고이력(격자매핑).csv', encoding='cp949')
df_grid   = gpd.read_file('data/raw/5.김해시_격자(100X100).geojson')
df_river  = gpd.read_file('data/raw/9.김해시_하천현황.geojson')

## 1. 기본 데이터 탐색

In [ ]:
print('=== CCTV 설치현황 ===')
print(f'총 CCTV 개수: {len(df_cctv)}대')
print(df_cctv.head())
print(df_cctv.dtypes)

In [ ]:
print('=== 112신고 이력 ===')
print(f'총 신고 건수: {len(df_police):,}건')
print(f'연도 범위: {df_police["year"].min()} ~ {df_police["year"].max()}')
print(df_police.head())

In [ ]:
print('=== 112신고 사건 유형별 건수 (상위 10) ===')
print(df_police['case_type'].value_counts().head(10))

In [ ]:
# 연도별 신고 건수 추이
yearly = df_police.groupby('year').size().reset_index(name='count')

plt.figure(figsize=(10, 4))
plt.bar(yearly['year'], yearly['count'], color='steelblue', alpha=0.8)
plt.title('연도별 112신고 건수')
plt.xlabel('연도')
plt.ylabel('신고 건수')
plt.xticks(yearly['year'])
plt.tight_layout()
plt.savefig('results/yearly_police_count.png', dpi=150)
plt.show()

## 2. CCTV 설치 전후 112신고 건수 비교

In [ ]:
# 격자별 연도별 CCTV 개수
cctv_grid = df_cctv.groupby(['gid', 'year']).size().reset_index(name='cctv_n_points')

# 격자별 연도별 112신고 건수
police_grid = df_police.groupby(['gid', 'year']).size().reset_index(name='police_n_points')

print('CCTV 격자 매핑:', cctv_grid.shape)
print('112신고 격자 매핑:', police_grid.shape)

In [ ]:
# 2019 → 2020년 CCTV 설치 효과 분석
# 2019년 CCTV 설치 격자와 2020년 신고 건수 비교
X = cctv_grid[cctv_grid['year'] == 2020][['gid', 'cctv_n_points']].rename(
    columns={'cctv_n_points': 'cctv_n_points_x'}
)
X['year_x'] = 2020

Y_before = police_grid[police_grid['year'] == 2019][['gid', 'police_n_points']].rename(
    columns={'police_n_points': 'police_n_points_x'}
)
Y_before['year_x'] = 2019

Y_after = police_grid[police_grid['year'] == 2020][['gid', 'police_n_points']].rename(
    columns={'police_n_points': 'police_n_points_y'}
)
Y_after['year_y'] = 2020

merged = X.merge(Y_before, on='gid', how='outer').merge(Y_after, on='gid', how='outer')
merged['police_n_points_x'] = merged['police_n_points_x'].fillna(0)
merged['police_n_points_y'] = merged['police_n_points_y'].fillna(0)
merged['gap'] = merged['police_n_points_y'] - merged['police_n_points_x']

print('=== 2020년 CCTV 설치 효과 (2019→2020년 신고 건수 변화) ===')
print(merged['gap'].describe())
print(f"\n평균 감소 건수: {merged['gap'].mean():.4f}건")

In [ ]:
# 2020 → 2021년 CCTV 설치 효과 분석
X2 = cctv_grid[cctv_grid['year'] == 2021][['gid', 'cctv_n_points']].rename(
    columns={'cctv_n_points': 'cctv_n_points_x'}
)
X2['year_x'] = 2021

Y2_before = police_grid[police_grid['year'] == 2020][['gid', 'police_n_points']].rename(
    columns={'police_n_points': 'police_n_points_x'}
)
Y2_after = police_grid[police_grid['year'] == 2021][['gid', 'police_n_points']].rename(
    columns={'police_n_points': 'police_n_points_y'}
)

merged2 = X2.merge(Y2_before, on='gid', how='outer').merge(Y2_after, on='gid', how='outer')
merged2['police_n_points_x'] = merged2['police_n_points_x'].fillna(0)
merged2['police_n_points_y'] = merged2['police_n_points_y'].fillna(0)
merged2['gap'] = merged2['police_n_points_y'] - merged2['police_n_points_x']

print('=== 2021년 CCTV 설치 효과 (2020→2021년 신고 건수 변화) ===')
print(merged2['gap'].describe())
print(f"\n평균 감소 건수: {merged2['gap'].mean():.4f}건")

## 3. 공간 시각화

In [ ]:
# CCTV 위치 시각화 (Folium)
# 좌표 컬럼명은 실제 데이터에 맞게 조정 필요
m = folium.Map(location=[35.228, 128.889], zoom_start=12, tiles='CartoDB dark_matter')

# CCTV 위치 마커
for _, row in df_cctv.dropna(subset=['위도', '경도']).iterrows():
    folium.CircleMarker(
        location=[row['위도'], row['경도']],
        radius=3,
        color='lime',
        fill=True,
        fill_opacity=0.7
    ).add_to(m)

# 112신고 히트맵
police_coords = df_police.dropna(subset=['위도', '경도'])[['위도', '경도']].values.tolist()
HeatMap(police_coords, radius=8, blur=10, min_opacity=0.3).add_to(m)

m.save('results/cctv_police_map.html')
print('지도 저장 완료: results/cctv_police_map.html')
m